In [6]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Set random seed so you get the same realistic data every time you run it
np.random.seed(76)
random.seed(76)

print("Starting data generation...")

# ==========================================
# 1. GENERATE VENDORS
# ==========================================
vendors_data = [
    {'vendor_id': 'V-1001', 'vendor_name': 'Apex SteelWorks', 'country': 'Germany', 'performance_rating': 4.8},
    {'vendor_id': 'V-1002', 'vendor_name': 'TechTronics Asia', 'country': 'China', 'performance_rating': 3.2}, # Low rating for a reason!
    {'vendor_id': 'V-1003', 'vendor_name': 'GreenEnergy Cells', 'country': 'South Korea', 'performance_rating': 4.5},
    {'vendor_id': 'V-1004', 'vendor_name': 'LocalPack USA', 'country': 'USA', 'performance_rating': 4.9},
    {'vendor_id': 'V-1005', 'vendor_name': 'FastPlastics', 'country': 'Mexico', 'performance_rating': 4.0}
]
vendors_df = pd.DataFrame(vendors_data)
vendors_df.to_csv('vendors_extract.csv', index=False)
print("Created vendors_extract.csv")

# ==========================================
# 2. GENERATE PURCHASE ORDERS (Headers)
# ==========================================
num_orders = 500
start_date = datetime(2023, 1, 1)

po_headers = []
for i in range(num_orders):
    po_number = f"PO-2023-{str(i+1).zfill(4)}"
    
    # Weighted vendor selection (TechTronics gets the most orders)
    vendor_id = np.random.choice(
        ['V-1001', 'V-1002', 'V-1003', 'V-1004', 'V-1005'], 
        p=[0.15, 0.40, 0.15, 0.10, 0.20]
    )
    
# Random order date in 2023
    random_days = random.randint(0, 360)
    order_date = start_date + timedelta(days=random_days)
    
# Base lead time (10 to 30 days)
    lead_time = random.randint(10, 30)
    expected_delivery = order_date + timedelta(days=lead_time)
    
# Simulate realistic actual deliveries based on vendor behavior
    if vendor_id == 'V-1002':
        # TechTronics is notoriously late (60% chance of delay, 5-20 days late)
        delay = random.randint(5, 20) if random.random() < 0.6 else random.randint(-2, 2)
    elif vendor_id == 'V-1004':
        # LocalPack is always on time or early
        delay = random.randint(-3, 0)
    else:
        # Others are generally on time, occasionally late
        delay = random.randint(1, 7) if random.random() < 0.2 else random.randint(-2, 2)
        
    actual_delivery = expected_delivery + timedelta(days=delay)
    
# Determine Status
    if actual_delivery > expected_delivery:
        status = 'Delayed'
    else:
        status = 'Delivered On Time'
        
    po_headers.append({
        'po_number': po_number,
        'vendor_id': vendor_id,
        'order_date': order_date.strftime('%Y-%m-%d'),
        'expected_delivery_date': expected_delivery.strftime('%Y-%m-%d'),
        'actual_delivery_date': actual_delivery.strftime('%Y-%m-%d'),
        'po_status': status
    })

po_headers_df = pd.DataFrame(po_headers)
po_headers_df.to_csv('po_headers_extract.csv', index=False)
print("Created po_headers_extract.csv")

# ==========================================
# 3. GENERATE PO ITEMS (Line Items)
# ==========================================

# Material catalog with realistic base prices
materials = {
    'MAT-001': {'name': 'Steel Sheet', 'base_price': 50.00, 'vendor': 'V-1001'},
    'MAT-002': {'name': 'Microcontroller', 'base_price': 4.50, 'vendor': 'V-1002'},
    'MAT-003': {'name': 'Capacitor Batch', 'base_price': 1.20, 'vendor': 'V-1002'},
    'MAT-004': {'name': 'Lithium Battery', 'base_price': 45.00, 'vendor': 'V-1003'},
    'MAT-005': {'name': 'Cardboard Box', 'base_price': 0.80, 'vendor': 'V-1004'},
    'MAT-006': {'name': 'Plastic Casing', 'base_price': 2.50, 'vendor': 'V-1005'}
}

po_items = []
item_id_counter = 1

for _, po in po_headers_df.iterrows():
    v_id = po['vendor_id']
    
    # Find materials supplied by this specific vendor
    available_materials = [m_id for m_id, data in materials.items() if data['vendor'] == v_id]
    
    # Ensure we don't try to sample more items than the vendor actually sells
    max_items_to_buy = min(3, len(available_materials))
    num_items = random.randint(1, max_items_to_buy)
    
    # Select materials for this order
    selected_materials = random.sample(available_materials, num_items)
    
    for mat_id in selected_materials:
        # Fluctuate the price slightly around the base price to simulate market changes
        base_price = materials[mat_id]['base_price']
        price_variance = base_price * random.uniform(-0.05, 0.05)
        unit_price = round(base_price + price_variance, 2)
        
        # Determine quantity (cheap items are bought in bulk)
        if base_price < 5.00:
            qty = random.randint(1000, 5000)
        else:
            qty = random.randint(10, 200)
            
        po_items.append({
            'item_id': item_id_counter,
            'po_number': po['po_number'],
            'material_id': mat_id,
            'quantity': qty,
            'unit_price': unit_price
        })
        item_id_counter += 1

po_items_df = pd.DataFrame(po_items)
po_items_df.to_csv('po_items_extract.csv', index=False)
print("Created po_items_extract.csv")
print("All files generated successfully! You are ready to run the ETL pipeline.")

Starting data generation...
Created vendors_extract.csv
Created po_headers_extract.csv
Created po_items_extract.csv
All files generated successfully! You are ready to run the ETL pipeline.


In [9]:
import pandas as pd

print("--- Data Ingestion ---")
# 1. Read the CSVs into Pandas DataFrames
vendors_df = pd.read_csv('vendors_extract.csv')
po_headers_df = pd.read_csv('po_headers_extract.csv')
po_items_df = pd.read_csv('po_items_extract.csv')

# 2. Clean Data: Convert text to Datetime objects
po_headers_df['order_date'] = pd.to_datetime(po_headers_df['order_date'])
po_headers_df['expected_delivery_date'] = pd.to_datetime(po_headers_df['expected_delivery_date'])
po_headers_df['actual_delivery_date'] = pd.to_datetime(po_headers_df['actual_delivery_date'])

# 3. Calculate 'total_value' in Python (since we don't have SQL's GENERATED ALWAYS)
po_items_df['total_value'] = po_items_df['quantity'] * po_items_df['unit_price']

--- Data Ingestion ---


In [10]:
print("\n--- Insight 1: Top Vendors by Spend ---")
# Python equivalent of JOINing the three tables
merged_df = po_items_df.merge(po_headers_df, on='po_number').merge(vendors_df, on='vendor_id')

# Python equivalent of GROUP BY and SUM
spend_by_vendor = merged_df.groupby('vendor_name')['total_value'].sum().reset_index()
spend_by_vendor = spend_by_vendor.sort_values(by='total_value', ascending=False)

# Formatting the output to look like currency
spend_by_vendor['total_value'] = spend_by_vendor['total_value'].apply(lambda x: f"${x:,.2f}")
print(spend_by_vendor.to_string(index=False))


print("\n--- Insight 2: Delayed Deliveries per Vendor ---")
# Python equivalent of WHERE clause (filtering)
delayed_orders = po_headers_df[po_headers_df['actual_delivery_date'] > po_headers_df['expected_delivery_date']]

# Merge with vendors to get the names
delayed_with_names = delayed_orders.merge(vendors_df, on='vendor_id')

# Python equivalent of GROUP BY and COUNT
delayed_counts = delayed_with_names.groupby('vendor_name').size().reset_index(name='total_delayed_orders')
delayed_counts = delayed_counts.sort_values(by='total_delayed_orders', ascending=False)

print(delayed_counts.to_string(index=False))


--- Insight 1: Top Vendors by Spend ---
      vendor_name   total_value
 TechTronics Asia $2,549,631.16
     FastPlastics   $785,784.98
  Apex SteelWorks   $506,831.26
GreenEnergy Cells   $378,074.83
    LocalPack USA    $97,929.87

--- Insight 2: Delayed Deliveries per Vendor ---
      vendor_name  total_delayed_orders
 TechTronics Asia                   142
     FastPlastics                    60
  Apex SteelWorks                    48
GreenEnergy Cells                    40
